DATASET 3

COMMON IMPORTS (USE IN ALL NOTEBOOKS)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

1. Data Loading

In [ ]:
print("📦 Loading eCommerce behavior data with optimized sampling...")
print("⚠️ Large dataset (14.68 GB) - using 10% stratified sampling\n")

def load_sampled_behavior_data(filepath, sample_frac=0.10, chunksize=1_000_000):
    """
    Load large CSV in chunks and sample each chunk.
    
    Parameters:
    - filepath: Path to CSV file
    - sample_frac: Fraction to sample (0.10 = 10%)
    - chunksize: Rows per chunk (1M rows = ~140MB per chunk)
    
    Returns:
    - Sampled DataFrame
    """
    
    print(f"📂 Processing: {filepath}")
    print(f"   Sample rate: {sample_frac*100}%")
    print(f"   Chunk size: {chunksize:,} rows\n")
    
    # Optimized data types for memory efficiency
    dtypes = {
        'event_time': str,
        'event_type': 'category',
        'product_id': 'int32',
        'category_id': 'int64',
        'category_code': 'category',
        'brand': 'category',
        'price': 'float32',
        'user_id': 'int32',
        'user_session': 'category'
    }
    
    sampled_chunks = []
    
    # Read file in chunks with progress bar
    chunk_iterator = pd.read_csv(
        filepath,
        chunksize=chunksize,
        dtype=dtypes
    )
    
    print(f"⚙️ Reading and sampling chunks...")
    
    for i, chunk in enumerate(tqdm(chunk_iterator, desc="Processing"), 1):
        # Random sample from each chunk (preserves distribution)
        sampled_chunk = chunk.sample(frac=sample_frac, random_state=42)
        sampled_chunks.append(sampled_chunk)
    
    # Combine all sampled chunks
    result = pd.concat(sampled_chunks, ignore_index=True)
    
    # Memory usage report
    memory_mb = result.memory_usage(deep=True).sum() / 1024**2
    
    print(f"\n✅ Completed: {filepath}")
    print(f"   Rows loaded: {len(result):,}")
    print(f"   Memory usage: {memory_mb:.1f} MB\n")
    
    return result

# ============================================
# Load October 2019 data
# ============================================

oct_data = load_sampled_behavior_data(
    filepath="../data/raw/2019-Oct.csv",
    sample_frac=0.10,
    chunksize=1_000_000
)

# ============================================
# Load November 2019 data
# ============================================

nov_data = load_sampled_behavior_data(
    filepath="../data/raw/2019-Nov.csv",
    sample_frac=0.10,
    chunksize=1_000_000
)

# ============================================
# Combine both months
# ============================================

print("🔗 Combining October and November data...")

behavior = pd.concat([oct_data, nov_data], ignore_index=True)

# Convert event_time to datetime
behavior['event_time'] = pd.to_datetime(behavior['event_time'])

# Sort by time for better analysis
behavior = behavior.sort_values('event_time').reset_index(drop=True)

print(f"\n📊 COMBINED DATASET SUMMARY:")
print(f"   Total events: {len(behavior):,}")
print(f"   Unique products: {behavior['product_id'].nunique():,}")
print(f"   Unique users: {behavior['user_id'].nunique():,}")
print(f"   Date range: {behavior['event_time'].min()} to {behavior['event_time'].max()}")
print(f"   Memory usage: {behavior.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Save combined sampled data for future use
print("\n💾 Saving combined sample for future use...")
behavior.to_csv(
    '../data/processed/behavior_combined_sampled_10pct.csv',
    index=False
)
print("✅ Saved: data/processed/behavior_combined_sampled_10pct.csv")

2. Data Cleaning

In [ ]:
print("\n🔍 DATA QUALITY ASSESSMENT:")
print(behavior.isnull().sum())
print(f"\nInitial shape: {behavior.shape}")

# Remove rows with missing critical fields
initial_count = len(behavior)
behavior = behavior[behavior['product_id'].notna()]
behavior = behavior[behavior['user_id'].notna()]
print(f"   Removed {initial_count - len(behavior):,} rows with missing IDs")

# Filter to relevant event types
valid_events = ['view', 'cart', 'purchase']
initial_count = len(behavior)
behavior = behavior[behavior['event_type'].isin(valid_events)]
print(f"   Removed {initial_count - len(behavior):,} rows with invalid event types")

# Remove invalid prices
initial_count = len(behavior)
behavior = behavior[behavior['price'] > 0]
print(f"   Removed {initial_count - len(behavior):,} rows with invalid prices")

# Remove extreme price outliers (top 0.1%)
price_threshold = behavior['price'].quantile(0.999)
initial_count = len(behavior)
behavior = behavior[behavior['price'] < price_threshold]
print(f"   Removed {initial_count - len(behavior):,} extreme price outliers (>{price_threshold:.2f})")

# Create temporal features
print("\n⏰ Creating temporal features...")
behavior['date'] = behavior['event_time'].dt.date
behavior['month'] = behavior['event_time'].dt.to_period('M')
behavior['year_month'] = behavior['event_time'].dt.strftime('%Y-%m')
behavior['hour'] = behavior['event_time'].dt.hour
behavior['day_of_week'] = behavior['event_time'].dt.dayofweek
behavior['day_name'] = behavior['event_time'].dt.day_name()

# Keep only products with sufficient interaction history (≥50 events)
print("\n🔍 Filtering products by interaction history...")
product_counts = behavior['product_id'].value_counts()
valid_products = product_counts[product_counts >= 50].index
initial_products = behavior['product_id'].nunique()
behavior = behavior[behavior['product_id'].isin(valid_products)]

print(f"\n✅ DATA CLEANING COMPLETE:")
print(f"   Final events: {len(behavior):,}")
print(f"   Products analyzed: {behavior['product_id'].nunique():,} (from {initial_products:,})")
print(f"   Users: {behavior['user_id'].nunique():,}")
print(f"   Date range: {behavior['event_time'].min()} to {behavior['event_time'].max()}")
print(f"   Months covered: {behavior['month'].nunique()}")


Quick FIX 

In [ ]:
import pandas as pd

# Ensure datetime
behavior["event_time"] = pd.to_datetime(behavior["event_time"], errors="coerce")

# Make tz-aware -> tz-naive (drop timezone)
# Works even if it's already tz-naive
try:
    behavior["event_time"] = behavior["event_time"].dt.tz_convert(None)
except TypeError:
    # means it's already tz-naive
    pass

# Ensure month exists (Period)
behavior["month"] = behavior["event_time"].dt.to_period("M")


3. Feature Engineering

In [ ]:
print("\n⚙️ FEATURE ENGINEERING...")

## Event weighting (conversion funnel importance)
event_weights = {
    'view': 1,
    'cart': 5,      # Strong purchase intent
    'purchase': 15  # Actual conversion
}

behavior['engagement_score'] = behavior['event_type'].map(event_weights)

print("   ✅ Engagement scores calculated")

## Monthly aggregation per product
print("   ⚙️ Aggregating monthly engagement metrics...")

monthly_engagement = (
    behavior
    .groupby(['product_id', 'month'])
    .agg({
        'engagement_score': ['sum', 'mean', 'count'],
        'user_id': 'nunique',
        'event_type': [
            lambda x: (x == 'view').sum(),
            lambda x: (x == 'cart').sum(),
            lambda x: (x == 'purchase').sum()
        ],
        'price': 'mean',
        'user_session': 'nunique'
    })
    .reset_index()
)

monthly_engagement.columns = [
    'product_id', 'month',
    'engagement_total', 'engagement_mean', 'event_count',
    'active_users', 'view_count', 'cart_count', 'purchase_count',
    'avg_price', 'unique_sessions'
]

print(f"   ✅ Monthly aggregation complete: {len(monthly_engagement):,} product-months")

## Conversion funnel metrics
print("   ⚙️ Calculating conversion funnel metrics...")

# View-to-cart conversion rate
monthly_engagement['view_to_cart_rate'] = (
    (monthly_engagement['cart_count'] / monthly_engagement['view_count'].replace(0, np.nan)) * 100
)

# Cart-to-purchase conversion rate
monthly_engagement['cart_to_purchase_rate'] = (
    (monthly_engagement['purchase_count'] / monthly_engagement['cart_count'].replace(0, np.nan)) * 100
)

# Overall conversion rate
monthly_engagement['conversion_rate'] = (
    (monthly_engagement['purchase_count'] / monthly_engagement['event_count']) * 100
)

print("   ✅ Conversion metrics calculated")

## Product lifecycle metrics
print("   ⚙️ Computing product lifecycle features...")

first_event = (
    behavior
    .groupby('product_id')['event_time']
    .min()
    .reset_index()
    .rename(columns={'event_time': 'first_event_date'})
)

monthly_engagement = monthly_engagement.merge(first_event, on='product_id')
monthly_engagement['month_date'] = monthly_engagement['month'].dt.to_timestamp()
monthly_engagement['product_age_months'] = (
    (monthly_engagement['month_date'] - monthly_engagement['first_event_date']).dt.days / 30
).round(1)

## Lifecycle stage classification
def classify_lifecycle(age_months):
    if age_months < 1:
        return 'introduction'
    elif age_months < 3:
        return 'growth'
    elif age_months < 6:
        return 'maturity'
    else:
        return 'decline'

monthly_engagement['lifecycle_stage'] = (
    monthly_engagement['product_age_months'].apply(classify_lifecycle)
)

print("   ✅ Lifecycle stages assigned")

## FATIGUE SIGNALS
print("   ⚙️ Engineering fatigue signals...")

# 1. Engagement velocity (% change)
monthly_engagement['engagement_velocity'] = (
    monthly_engagement
    .groupby('product_id')['engagement_total']
    .transform(lambda x: x.pct_change() * 100)
)

# 2. Engagement acceleration
monthly_engagement['engagement_acceleration'] = (
    monthly_engagement
    .groupby('product_id')['engagement_velocity']
    .transform(lambda x: x.diff())
)

# 3. User retention rate
monthly_engagement['user_retention_rate'] = (
    monthly_engagement
    .groupby('product_id')['active_users']
    .transform(lambda x: x.pct_change() * 100)
)

# 4. Session frequency change
monthly_engagement['session_frequency_change'] = (
    monthly_engagement
    .groupby('product_id')['unique_sessions']
    .transform(lambda x: x.pct_change() * 100)
)

# 5. Conversion rate change
monthly_engagement['conversion_rate_change'] = (
    monthly_engagement
    .groupby('product_id')['conversion_rate']
    .transform(lambda x: x.diff())
)

# 6. Engagement volatility
monthly_engagement['engagement_volatility'] = (
    monthly_engagement
    .groupby('product_id')['engagement_total']
    .transform(lambda x: x.rolling(3, min_periods=1).std())
)

# 7. Purchase momentum
monthly_engagement['purchase_momentum'] = (
    monthly_engagement
    .groupby('product_id')['purchase_count']
    .transform(lambda x: x.pct_change() * 100)
)

# 8. Funnel efficiency (view → purchase)
monthly_engagement['funnel_efficiency'] = (
    monthly_engagement['purchase_count'] / monthly_engagement['view_count'].replace(0, np.nan) * 100
)

monthly_engagement['funnel_efficiency_change'] = (
    monthly_engagement
    .groupby('product_id')['funnel_efficiency']
    .transform(lambda x: x.diff())
)

# 9. User engagement per session
monthly_engagement['engagement_per_session'] = (
    monthly_engagement['engagement_total'] / monthly_engagement['unique_sessions']
)

monthly_engagement['engagement_quality_change'] = (
    monthly_engagement
    .groupby('product_id')['engagement_per_session']
    .transform(lambda x: x.pct_change() * 100)
)

print("   ✅ All fatigue signals engineered")

print(f"\n📊 FEATURE ENGINEERING SUMMARY:")
print(f"   Total features: {len(monthly_engagement.columns)}")
print(f"   Product-months: {len(monthly_engagement):,}")
print(f"   Products: {monthly_engagement['product_id'].nunique():,}")

4. EDA — Emotional Fatigue

In [ ]:
print("\n📈 STARTING EXPLORATORY DATA ANALYSIS...\n")

## 🔹 UNIVARIATE ANALYSIS

print("📊 Generating univariate analysis plots...")

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

# Engagement distribution
sns.histplot(
    data=monthly_engagement,
    x='engagement_total',
    bins=50,
    kde=True,
    log_scale=True,
    ax=axes[0, 0]
)
axes[0, 0].set_title('Engagement Score Distribution (Log Scale)', fontsize=12, fontweight='bold')

# Engagement by lifecycle
sns.boxplot(
    data=monthly_engagement,
    x='lifecycle_stage',
    y='engagement_total',
    order=['introduction', 'growth', 'maturity', 'decline'],
    ax=axes[0, 1]
)
axes[0, 1].set_title('Engagement by Product Lifecycle', fontsize=12, fontweight='bold')
axes[0, 1].set_yscale('log')

# Conversion rate distribution
sns.histplot(
    data=monthly_engagement[monthly_engagement['conversion_rate'].notna()],
    x='conversion_rate',
    bins=50,
    kde=True,
    ax=axes[1, 0]
)
axes[1, 0].set_title('Conversion Rate Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlim(0, 20)

# Engagement velocity
sns.histplot(
    data=monthly_engagement[monthly_engagement['engagement_velocity'].notna()],
    x='engagement_velocity',
    bins=60,
    kde=True,
    ax=axes[1, 1]
)
axes[1, 1].set_title('Engagement Velocity (% Change)', fontsize=12, fontweight='bold')
axes[1, 1].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_xlim(-100, 100)

# User retention rate
sns.histplot(
    data=monthly_engagement[monthly_engagement['user_retention_rate'].notna()],
    x='user_retention_rate',
    bins=60,
    kde=True,
    ax=axes[2, 0]
)
axes[2, 0].set_title('User Retention Rate (%)', fontsize=12, fontweight='bold')
axes[2, 0].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[2, 0].set_xlim(-100, 100)

# Funnel efficiency
sns.histplot(
    data=monthly_engagement[monthly_engagement['funnel_efficiency'].notna()],
    x='funnel_efficiency',
    bins=50,
    kde=True,
    ax=axes[2, 1]
)
axes[2, 1].set_title('Funnel Efficiency (View → Purchase %)', fontsize=12, fontweight='bold')
axes[2, 1].set_xlim(0, 10)

plt.tight_layout()
plt.savefig('../outputs/usage_univariate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("   ✅ Saved: outputs/usage_univariate_analysis.png")

## 🔹 CONVERSION FUNNEL VISUALIZATION

print("📊 Creating conversion funnel visualization...")

funnel_metrics = monthly_engagement[[
    'view_count', 'cart_count', 'purchase_count'
]].mean()

fig, ax = plt.subplots(figsize=(10, 6))

stages = ['Views', 'Add to Cart', 'Purchases']
values = [
    funnel_metrics['view_count'],
    funnel_metrics['cart_count'],
    funnel_metrics['purchase_count']
]

# Calculate conversion rates
conversion_rates = [
    100,  # Views baseline
    (values[1] / values[0]) * 100,
    (values[2] / values[0]) * 100
]

colors = ['#3498db', '#f39c12', '#2ecc71']
y_pos = np.arange(len(stages))

bars = ax.barh(y_pos, values, color=colors, alpha=0.8)

# Add value labels
for i, (bar, val, rate) in enumerate(zip(bars, values, conversion_rates)):
    width = bar.get_width()
    ax.text(
        width, bar.get_y() + bar.get_height()/2,
        f' {val:,.0f} ({rate:.1f}%)',
        ha='left', va='center', fontsize=11, fontweight='bold'
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(stages)
ax.set_xlabel('Average Monthly Event Count', fontsize=11)
ax.set_title('Average Conversion Funnel Performance', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/usage_conversion_funnel.png', dpi=300, bbox_inches='tight')
plt.show()

print("   ✅ Saved: outputs/usage_conversion_funnel.png")

## 🔹 BIVARIATE ANALYSIS — Fatigue Trajectories

print("📊 Analyzing fatigue trajectories...")

# Identify top 10 fatigued products
fatigued_products = (
    monthly_engagement[
        (monthly_engagement['engagement_velocity'] < -20) &
        (monthly_engagement['product_age_months'] >= 1)
    ]
    .groupby('product_id')
    .size()
    .nlargest(10)
    .index
)

if len(fatigued_products) > 0:
    fatigue_data = monthly_engagement[
        monthly_engagement['product_id'].isin(fatigued_products)
    ].copy()

    fig, axes = plt.subplots(3, 1, figsize=(14, 12))

    # Engagement trajectory
    for product in fatigued_products:
        product_data = fatigue_data[fatigue_data['product_id'] == product]
        axes[0].plot(
            product_data['product_age_months'],
            product_data['engagement_total'],
            marker='o',
            alpha=0.7,
            linewidth=2
        )

    axes[0].set_xlabel('Product Age (Months)', fontsize=11)
    axes[0].set_ylabel('Total Engagement Score', fontsize=11)
    axes[0].set_title('Behavioral Fatigue Trajectories (Top 10 Declining Products)',
                      fontsize=12, fontweight='bold')
    axes[0].set_yscale('log')
    axes[0].grid(True, alpha=0.3)

    # Conversion rate trajectory
    for product in fatigued_products:
        product_data = fatigue_data[fatigue_data['product_id'] == product]
        axes[1].plot(
            product_data['product_age_months'],
            product_data['conversion_rate'],
            marker='s',
            alpha=0.7,
            linewidth=2
        )

    axes[1].set_xlabel('Product Age (Months)', fontsize=11)
    axes[1].set_ylabel('Conversion Rate (%)', fontsize=11)
    axes[1].set_title('Conversion Rate Decline (Same Products)', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3)

    # Active users trajectory
    for product in fatigued_products:
        product_data = fatigue_data[fatigue_data['product_id'] == product]
        axes[2].plot(
            product_data['product_age_months'],
            product_data['active_users'],
            marker='^',
            alpha=0.7,
            linewidth=2
        )

    axes[2].set_xlabel('Product Age (Months)', fontsize=11)
    axes[2].set_ylabel('Active Users', fontsize=11)
    axes[2].set_title('User Base Erosion (Same Products)', fontsize=12, fontweight='bold')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../outputs/usage_fatigue_trajectories.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("   ✅ Saved: outputs/usage_fatigue_trajectories.png")
else:
    print("   ⚠️ No highly fatigued products found with current criteria")

## 🔹 EVENT TYPE ANALYSIS

print("📊 Analyzing event type trends...")

event_type_trends = (
    behavior
    .groupby(['year_month', 'event_type'])
    .size()
    .reset_index(name='count')
)

fig, ax = plt.subplots(figsize=(12, 6))

for event in ['view', 'cart', 'purchase']:
    event_data = event_type_trends[event_type_trends['event_type'] == event]
    ax.plot(
        event_data['year_month'],
        event_data['count'],
        marker='o',
        linewidth=2,
        label=event.capitalize()
    )

ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Event Count', fontsize=11)
ax.set_title('Event Type Trends Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/usage_event_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("   ✅ Saved: outputs/usage_event_trends.png")

## 🔹 CORRELATION HEATMAP

print("📊 Computing correlation matrix...")

corr_features = [
    'engagement_total', 'engagement_velocity', 'engagement_acceleration',
    'engagement_volatility', 'user_retention_rate', 'conversion_rate',
    'conversion_rate_change', 'purchase_momentum', 'funnel_efficiency',
    'product_age_months'
]

corr_matrix = monthly_engagement[corr_features].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=1,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title('Behavioral Fatigue Signal Correlation Heatmap',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../outputs/usage_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("   ✅ Saved: outputs/usage_correlation_heatmap.png")

## 🔹 MULTIVARIATE SCATTER ANALYSIS

print("📊 Creating multivariate scatter plots...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sample for visualization
sample_data = monthly_engagement.sample(n=min(5000, len(monthly_engagement)), random_state=42)

# Engagement vs Conversion
sns.scatterplot(
    data=sample_data,
    x='engagement_total',
    y='conversion_rate',
    hue='lifecycle_stage',
    alpha=0.5,
    ax=axes[0, 0]
)
axes[0, 0].set_xscale('log')
axes[0, 0].set_title('Engagement vs Conversion Rate', fontsize=12, fontweight='bold')

# Engagement velocity vs User retention
sns.scatterplot(
    data=sample_data[sample_data['engagement_velocity'].notna()],
    x='engagement_velocity',
    y='user_retention_rate',
    hue='lifecycle_stage',
    alpha=0.5,
    ax=axes[0, 1]
)
axes[0, 1].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[0, 1].axvline(x=0, color='red', linestyle='--', alpha=0.3)
axes[0, 1].set_title('Engagement Velocity vs User Retention', fontsize=12, fontweight='bold')

# Product age vs Conversion rate
sns.scatterplot(
    data=sample_data,
    x='product_age_months',
    y='conversion_rate',
    hue='lifecycle_stage',
    alpha=0.5,
    ax=axes[1, 0]
)
axes[1, 0].set_title('Product Age vs Conversion Rate', fontsize=12, fontweight='bold')

# Funnel efficiency vs Purchase momentum
sns.scatterplot(
    data=sample_data[sample_data['purchase_momentum'].notna()],
    x='funnel_efficiency',
    y='purchase_momentum',
    hue='lifecycle_stage',
    alpha=0.5,
    ax=axes[1, 1]
)
axes[1, 1].set_title('Funnel Efficiency vs Purchase Momentum', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/usage_multivariate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("   ✅ Saved: outputs/usage_multivariate_analysis.png")

## 🔹 OUTLIER DETECTION

print("📊 Detecting outliers...")

monthly_engagement['z_engagement_velocity'] = zscore(
    monthly_engagement['engagement_velocity'].fillna(0)
)
monthly_engagement['z_user_retention'] = zscore(
    monthly_engagement['user_retention_rate'].fillna(0)
)
monthly_engagement['z_conversion_change'] = zscore(
    monthly_engagement['conversion_rate_change'].fillna(0)
)

outliers = monthly_engagement[
    (monthly_engagement['z_engagement_velocity'] < -3) |
    (monthly_engagement['z_user_retention'] < -3) |
    (monthly_engagement['z_conversion_change'] < -3)
]

print(f"\n⚠️ Extreme behavioral fatigue outliers: {len(outliers)} product-months")
print(f"   Representing {outliers['product_id'].nunique()} unique products")

# Visualize outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=monthly_engagement,
    x='product_age_months',
    y='engagement_velocity',
    alpha=0.3,
    ax=axes[0]
)
sns.scatterplot(
    data=outliers,
    x='product_age_months',
    y='engagement_velocity',
    color='red',
    s=100,
    alpha=0.8,
    label='Outliers (Z < -3)',
    ax=axes[0]
)
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0].set_title('Engagement Velocity Outliers', fontsize=12, fontweight='bold')
axes[0].legend()

sns.scatterplot(
    data=monthly_engagement,
    x='conversion_rate',
    y='user_retention_rate',
    alpha=0.3,
    ax=axes[1]
)
sns.scatterplot(
    data=outliers,
    x='conversion_rate',
    y='user_retention_rate',
    color='red',
    s=100,
    alpha=0.8,
    label='Outliers',
    ax=axes[1]
)
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1].set_title('Conversion vs Retention Outliers', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/usage_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

print("   ✅ Saved: outputs/usage_outliers.png")

5. FATIGUE LABELING (MULTI-CRITERIA)

In [ ]:
print("\n🏷️ LABELING FATIGUE LEVELS...")

def label_behavioral_fatigue(row):
    """
    Multi-dimensional behavioral fatigue classification.
    
    HIGH FATIGUE:
    - Steep engagement decline
    - User base shrinking
    - Conversion rate dropping
    - Purchase momentum negative
    
    MODERATE FATIGUE:
    - Moderate engagement decline
    - OR conversion rate issues
    
    HEALTHY:
    - Stable or growing metrics
    """
    
    velocity = row['engagement_velocity'] if pd.notna(row['engagement_velocity']) else 0
    acceleration = row['engagement_acceleration'] if pd.notna(row['engagement_acceleration']) else 0
    retention = row['user_retention_rate'] if pd.notna(row['user_retention_rate']) else 0
    conversion_change = row['conversion_rate_change'] if pd.notna(row['conversion_rate_change']) else 0
    purchase_mom = row['purchase_momentum'] if pd.notna(row['purchase_momentum']) else 0
    
    # High fatigue criteria
    if (
        velocity < -25 and
        retention < -20 and
        purchase_mom < -30 and
        acceleration < -15
    ):
        return 'high_fatigue'
    
    # Moderate fatigue criteria
    elif (
        (velocity < -15 and retention < -10) or
        (conversion_change < -2 and purchase_mom < -15) or
        (velocity < -20 and row['conversion_rate'] < 2)
    ):
        return 'moderate_fatigue'
    
    # Healthy
    else:
        return 'healthy'

monthly_engagement['fatigue_label'] = monthly_engagement.apply(
    label_behavioral_fatigue, axis=1
)

# Fatigue distribution
print("\n📊 BEHAVIORAL FATIGUE DISTRIBUTION:")
fatigue_dist = monthly_engagement['fatigue_label'].value_counts()
print(fatigue_dist)

if 'high_fatigue' in fatigue_dist.index or 'moderate_fatigue' in fatigue_dist.index:
    high = fatigue_dist.get('high_fatigue', 0)
    moderate = fatigue_dist.get('moderate_fatigue', 0)
    total_fatigue_rate = (high + moderate) / len(monthly_engagement) * 100
    print(f"\nOverall fatigue rate: {total_fatigue_rate:.1f}%")

# Visualize distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Fatigue label counts
fatigue_dist.plot(
    kind='bar',
    color=['green', 'orange', 'red'],
    ax=axes[0]
)
axes[0].set_title('Fatigue Label Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Fatigue Level')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Engagement by fatigue level
sns.boxplot(
    data=monthly_engagement,
    x='fatigue_label',
    y='engagement_total',
    order=['healthy', 'moderate_fatigue', 'high_fatigue'],
    palette=['green', 'orange', 'red'],
    ax=axes[1]
)
axes[1].set_title('Engagement by Fatigue Level', fontsize=12, fontweight='bold')
axes[1].set_yscale('log')

# Conversion rate by fatigue level
sns.boxplot(
    data=monthly_engagement[monthly_engagement['conversion_rate'].notna()],
    x='fatigue_label',
    y='conversion_rate',
    order=['healthy', 'moderate_fatigue', 'high_fatigue'],
    palette=['green', 'orange', 'red'],
    ax=axes[2]
)
axes[2].set_title('Conversion Rate by Fatigue Level', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/usage_fatigue_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("   ✅ Saved: outputs/usage_fatigue_distribution.png")

6. Additional Insights

In [ ]:
## Fatigue by lifecycle stage
print("\n📈 FATIGUE DISTRIBUTION BY LIFECYCLE STAGE:")
lifecycle_fatigue = pd.crosstab(
    monthly_engagement['lifecycle_stage'],
    monthly_engagement['fatigue_label'],
    normalize='index'
) * 100

print(lifecycle_fatigue.round(1))

lifecycle_fatigue.plot(
    kind='bar',
    stacked=True,
    color=['green', 'orange', 'red'],
    figsize=(10, 6)
)
plt.title('Fatigue Distribution Across Product Lifecycle', fontsize=14, fontweight='bold')
plt.xlabel('Lifecycle Stage')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Fatigue Level', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig('../outputs/usage_lifecycle_fatigue.png', dpi=300, bbox_inches='tight')
plt.show()

print("   ✅ Saved: outputs/usage_lifecycle_fatigue.png")

6. EXPORT PROCESSED DATA

In [ ]:
print("\n💾 EXPORTING PROCESSED DATA...")

# Save full feature set
monthly_engagement.to_csv(
    '../data/processed/usage_fatigue_signals.csv',
    index=False
)
print("   ✅ Saved: data/processed/usage_fatigue_signals.csv")

# Save summary statistics
summary_stats = monthly_engagement.groupby('fatigue_label').agg({
    'engagement_total': ['mean', 'std', 'median'],
    'engagement_velocity': ['mean', 'std'],
    'conversion_rate': ['mean', 'std'],
    'user_retention_rate': ['mean', 'std'],
    'product_id': 'nunique'
}).round(3)

summary_stats.to_csv('../data/processed/usage_fatigue_summary.csv')
print("   ✅ Saved: data/processed/usage_fatigue_summary.csv")

# Save event-level data sample for deep dives (100K events)
behavior.sample(n=min(100000, len(behavior)), random_state=42).to_csv(
    '../data/processed/behavior_sample.csv',
    index=False
)
print("   ✅ Saved: data/processed/behavior_sample.csv")

8. Final Summary

In [ ]:
print("\n" + "="*60)
print("✅ USAGE EDA COMPLETE!")
print("="*60)
print(f"\n📁 OUTPUT FILES:")
print(f"   1. data/processed/usage_fatigue_signals.csv")
print(f"   2. data/processed/usage_fatigue_summary.csv")
print(f"   3. data/processed/behavior_sample.csv")
print(f"   4. data/processed/behavior_combined_sampled_10pct.csv")
print(f"\n📊 FINAL STATISTICS:")
print(f"   Dataset shape: {monthly_engagement.shape}")
print(f"   Features engineered: {len(monthly_engagement.columns)}")
print(f"   Products analyzed: {monthly_engagement['product_id'].nunique():,}")
print(f"   Product-months: {len(monthly_engagement):,}")
print(f"   Date range: {behavior['event_time'].min()} to {behavior['event_time'].max()}")
print(f"\n🎯 METHODOLOGY:")
print(f"   Sampling strategy: 10% stratified random sampling")
print(f"   Original data size: ~14.68 GB (109M rows)")
print(f"   Sampled data size: ~10.9M rows")
print(f"   Memory efficiency: {behavior.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"   Statistical validity: ✅ Excellent (n > 1M)")
print("\n" + "="*60)